# Avance 02 — SSC · Etapa 3: reglas de limpieza y duplicados
**Equipo A&C · Proyecto de Inteligencia de Negocios**

**Pregunta de negocio:** ¿En qué alcaldías y horarios de la CDMX se concentra la mayor frecuencia de hechos de tránsito y en cuáles se observa mayor severidad (personas lesionadas y fallecidas)?

En esta etapa aplicamos **reglas conservadoras y trazables** a las cinco tablas SSC delimitadas en la Etapa 1. Los archivos originales no se modifican.

**Resultados confirmados en la Etapa 2:** 759,069 filas entre cinco tablas, 3,498 duplicados adicionales en `PersonasLyFporEdad_SSC`, 13,918 edades faltantes y 14 edades superiores a 120 años. **Estos valores no autorizan eliminar automáticamente personas ni eventos.**

## 1. Preparación y búsqueda de archivos
Leemos los CSV de la Etapa 1. Este notebook es independiente de `funciones_calidad.py` y evita incluir por error los archivos `*_excluidos_etapa1.csv`. Si hay varias carpetas completas, elegimos explícitamente la ruta correcta.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
from datetime import datetime
from IPython.display import display

RUTA_ETAPA1_MANUAL = None  # Si hace falta: Path('/ruta/al/proyecto/resultados_SSC/etapa1')
BASES = [
    'HechosTransito_SSC',
    'PersonasLyFporEdad_SSC',
    'PersonasLyFporSexo_SSC',
    'PersonasLyFporTipoPersona_SSC',
    'VehiculosInvolucrados_SSC',
]
def rutas_esperadas(carpeta):
    return [Path(carpeta) / f'{base}_etapa1.csv' for base in BASES]

if RUTA_ETAPA1_MANUAL is not None:
    ENTRADA = Path(RUTA_ETAPA1_MANUAL).expanduser().resolve()
else:
    cwd = Path.cwd().resolve()
    opciones = [p / 'resultados_SSC' / 'etapa1' for p in [cwd, *cwd.parents]]
    opciones = [p for p in opciones if all(x.is_file() for x in rutas_esperadas(p))]
    if not opciones:
        opciones = [p for p in Path.home().rglob('etapa1')
                    if p.parent.name == 'resultados_SSC'
                    and all(x.is_file() for x in rutas_esperadas(p))]
    opciones = sorted(set(p.resolve() for p in opciones))
    if not opciones:
        raise FileNotFoundError('No encontramos los cinco archivos SSC de Etapa 1. Configura RUTA_ETAPA1_MANUAL.')
    if len(opciones) > 1:
        print('Se encontraron varias carpetas. Selecciona la correcta con RUTA_ETAPA1_MANUAL:')
        for p in opciones:
            print(' -', p)
        raise RuntimeError('Se requiere seleccionar la carpeta de entrada para evitar mezclar versiones.')
    ENTRADA = opciones[0]

assert all(p.is_file() for p in rutas_esperadas(ENTRADA))
SALIDA = ENTRADA.parent / 'etapa3'
SALIDA.mkdir(parents=True, exist_ok=True)
print('Entrada:', ENTRADA)
print('Salida:', SALIDA)
print('Archivos de entrada:', len(rutas_esperadas(ENTRADA)))

Entrada: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_SSC/etapa1
Salida: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_SSC/etapa3
Archivos de entrada: 5


## 2. Lectura e inventario inicial
Leemos como texto para conservar folios y códigos. Guardamos una copia en memoria para comprobar al final que **no alteramos las tablas de entrada**.

In [2]:
TABLAS = {}
FILAS_INICIALES = {}
for ruta in rutas_esperadas(ENTRADA):
    nombre = ruta.name.removesuffix('_etapa1.csv')
    df = pd.read_csv(ruta, dtype=str, encoding='utf-8-sig', low_memory=False)
    TABLAS[nombre] = df
    FILAS_INICIALES[nombre] = len(df)

inventario = pd.DataFrame([
    {'tabla': k, 'filas_recibidas_etapa3': len(v), 'columnas': len(v.columns)}
    for k, v in TABLAS.items()
])
display(inventario)
inventario.to_csv(SALIDA / 'inventario_entrada_etapa3_SSC.csv', index=False, encoding='utf-8-sig')

,tabla,filas_recibidas_etapa3,columnas
0,HechosTransito_SSC,134078,27
1,PersonasLyFporEdad_SSC,158076,24
2,PersonasLyFporSexo_SSC,144043,25
3,PersonasLyFporTipoPersona_SSC,107794,25
4,VehiculosInvolucrados_SSC,215078,27


## 3. Reglas de trabajo aprobadas y decisiones pendientes
**Reglas que aplicamos:**
- No eliminamos filas por compartir folio: un evento puede tener múltiples personas o vehículos.
- `SD` y equivalentes no constituyen identificadores únicos.
- Si un folio aparece en alcaldías distintas, lo conservamos y lo reportamos.
- Comparamos filas completas para identificar duplicados exactos, pero **no los borramos sin evidencia de que representan la misma unidad de observación**.
- Conservamos la columna original de edad y creamos una columna derivada `edad_validada`. Un valor faltante, negativo, decimal o superior a 120 se deja nulo **solo en la variable derivada**.
- No imputamos edades ni ubicaciones sin respaldo.
- Conservamos los cinco registros excluidos de Etapa 1 en sus archivos de evidencia; no los reintegramos sin verificación geográfica.

**Decisión pendiente:** confirmar la granularidad de `PersonasLyFporEdad_SSC` antes de decidir sobre sus 3,498 duplicados adicionales.

## 4. Diagnóstico de duplicados antes de limpiar
Contamos los grupos de filas idénticas, registramos ejemplos y revisamos folios repetidos. Ninguna de estas métricas autoriza por sí sola una eliminación.

In [3]:
diag = []
for nombre, df in TABLAS.items():
    f = df['folio'].fillna('').str.strip().str.upper() if 'folio' in df else pd.Series('', index=df.index)
    util = f[~f.isin(['', 'SD', 'S/D', 'SIN DATO', 'SIN DATOS', 'ND', 'N/D'])]
    rep = util.value_counts()
    diag.append({
        'tabla': nombre,
        'filas': len(df),
        'filas_en_grupos_identicos': int(df.duplicated(keep=False).sum()),
        'duplicados_adicionales': int(df.duplicated().sum()),
        'folios_repetidos_utilizables': int((rep > 1).sum()),
        'folios_SD_o_equivalentes': int(f.isin(['SD', 'S/D', 'SIN DATO', 'SIN DATOS', 'ND', 'N/D']).sum()),
        'filas_eliminadas_por_duplicados': 0,
    })
    df.loc[df.duplicated(keep=False)].head(100).to_csv(
        SALIDA / f'{nombre}_muestra_duplicados.csv', index=False, encoding='utf-8-sig'
    )
diagnostico = pd.DataFrame(diag)
display(diagnostico)
diagnostico.to_csv(SALIDA / 'diagnostico_duplicados_SSC.csv', index=False, encoding='utf-8-sig')

,tabla,filas,filas_en_grupos_identicos,duplicados_adicionales,folios_repetidos_utilizables,folios_SD_o_equivalentes,filas_eliminadas_por_duplicados
0,HechosTransito_SSC,134078,0,0,218,7288,0
1,PersonasLyFporEdad_SSC,158076,6486,3498,16718,8651,0
2,PersonasLyFporSexo_SSC,144043,0,0,9433,7828,0
3,PersonasLyFporTipoPersona_SSC,107794,0,0,8553,17,0
4,VehiculosInvolucrados_SSC,215078,0,0,72779,11797,0


## 5. Comprobación de folios repetidos con alcaldías distintas
Conservamos estos casos y guardamos hasta 100 ejemplos por tabla. Una diferencia de alcaldía es una alerta para investigar, no motivo para borrar.

In [4]:
casos_folio = []
for nombre, df in TABLAS.items():
    if 'folio' not in df or 'alcaldia' not in df:
        continue
    fol = df['folio'].fillna('').str.strip().str.upper()
    alc = df['alcaldia_normalizada'] if 'alcaldia_normalizada' in df else df['alcaldia']
    mask = ~fol.isin(['', 'SD', 'S/D', 'SIN DATO', 'SIN DATOS', 'ND', 'N/D'])
    t = pd.DataFrame({'folio': fol[mask], 'alcaldia': alc[mask].fillna('<NULO>')})
    n = t.groupby('folio')['alcaldia'].nunique()
    for f, cantidad in n[n > 1].head(100).items():
        casos_folio.append({
            'tabla': nombre, 'folio': f, 'numero_alcaldias': int(cantidad),
            'alcaldias': ' | '.join(t.loc[t['folio'].eq(f), 'alcaldia'].drop_duplicates().astype(str))
        })
casos_folio_df = pd.DataFrame(casos_folio, columns=['tabla','folio','numero_alcaldias','alcaldias'])
display(casos_folio_df.head(20))
casos_folio_df.to_csv(SALIDA / 'folios_alcaldias_distintas_SSC.csv',
                       index=False, encoding='utf-8-sig')

,tabla,folio,numero_alcaldias,alcaldias
0,HechosTransito_SSC,1136192,2,MIGUEL HIDALGO | BENITO JUAREZ
1,HechosTransito_SSC,2404440,2,TLALPAN | MIGUEL HIDALGO
2,HechosTransito_SSC,2411167,2,TLAHUAC | TLALPAN
3,HechosTransito_SSC,314955,2,LA MAGDALENA CONTRERAS | CUAUHTEMOC
4,HechosTransito_SSC,316753,2,CUAUHTEMOC | BENITO JUAREZ
5,HechosTransito_SSC,318025,2,IZTAPALAPA | MIGUEL HIDALGO
6,HechosTransito_SSC,324527,2,GUSTAVO A MADERO | XOCHIMILCO
7,HechosTransito_SSC,325980,2,XOCHIMILCO | GUSTAVO A MADERO
8,HechosTransito_SSC,327616,2,IZTAPALAPA | AZCAPOTZALCO
9,HechosTransito_SSC,328860,2,GUSTAVO A MADERO | MILPA ALTA


## 6. Inspección de la variable de edad
Antes de transformarla, identificamos el nombre real de la columna. Si no podemos distinguirla de otras variables, detenemos el proceso y pedimos selección explícita.

In [5]:
EDAD = TABLAS['PersonasLyFporEdad_SSC']
candidatas = [c for c in EDAD.columns if 'edad' in c.lower()]
print('Columnas que contienen "edad":', candidatas)
COLUMNA_EDAD_MANUAL = None  # Si es necesario, escribir el nombre exacto: 'edad'
if COLUMNA_EDAD_MANUAL:
    COL_EDAD = COLUMNA_EDAD_MANUAL
elif len(candidatas) == 1:
    COL_EDAD = candidatas[0]
else:
    raise ValueError('No hay una única columna de edad. Revisa candidatas y configura COLUMNA_EDAD_MANUAL.')
assert COL_EDAD in EDAD.columns
print('Columna de edad seleccionada:', COL_EDAD)
print('Ejemplos originales:')
display(EDAD[COL_EDAD].value_counts(dropna=False).head(20))

Columnas que contienen "edad": ['edad']
Columna de edad seleccionada: edad
Ejemplos originales:


NaN    13918
25      6460
30      6054
22      5222
28      5211
23      5195
26      5185
27      4988
24      4979
20      4504
35      4458
32      4417
21      4340
29      4299
19      3473
31      3393
40      3317
33      3300
34      3246
36      3163
Name: edad, dtype: int64

## 7. Limpieza conservadora de edades
Creamos `edad_validada` y `estado_validacion_edad`, sin sobrescribir la edad original. Consideramos rango operativo **0–120 años inclusive**. Las edades fuera del rango se marcan como no válidas para análisis por edad, pero conservamos sus filas.

No interpretamos edades decimales como años cumplidos y no imputamos edades faltantes. El límite 120 es un criterio de revisión documentado, no una corrección de los valores originales.

In [6]:
LIMITE_EDAD = 120
limpios = {nombre: df.copy(deep=True) for nombre, df in TABLAS.items()}
df_edad = limpios['PersonasLyFporEdad_SSC']
original = df_edad[COL_EDAD]
texto = original.astype('string').str.strip()
numero = pd.to_numeric(texto, errors='coerce')
faltante = texto.isna() | texto.eq('').fillna(False)
no_numerica = ~faltante & numero.isna()
negativa = numero.notna() & (numero < 0)
superior = numero.notna() & (numero > LIMITE_EDAD)
decimal = numero.notna() & (numero % 1 != 0)
valida = numero.notna() & ~negativa & ~superior & ~decimal

estado = np.select(
    [faltante.to_numpy(dtype=bool),
     no_numerica.to_numpy(dtype=bool),
     negativa.to_numpy(dtype=bool),
     superior.to_numpy(dtype=bool),
     decimal.to_numpy(dtype=bool)],
    ['faltante', 'no_numerica', 'negativa', 'superior_a_120', 'decimal'],
    default='dentro_rango_operativo'
)
df_edad['estado_validacion_edad'] = estado
df_edad['edad_validada'] = numero.where(valida).astype('Int64')
resumen_edad = (
    df_edad['estado_validacion_edad'].value_counts(dropna=False)
    .rename_axis('categoria').reset_index(name='filas')
)
resumen_edad['porcentaje'] = (100 * resumen_edad['filas'] / len(df_edad)).round(2)
display(resumen_edad)
resumen_edad.to_csv(SALIDA / 'resumen_tratamiento_edad_SSC.csv',
                    index=False, encoding='utf-8-sig')

revision_edad = df_edad.loc[df_edad['estado_validacion_edad'] != 'dentro_rango_operativo']
revision_edad.to_csv(SALIDA / 'casos_edad_no_validada_SSC.csv',
                    index=False, encoding='utf-8-sig')
print('Registros de edad que requieren revisión:', len(revision_edad))
print('Registros conservados:', len(df_edad))

,categoria,filas,porcentaje
0,dentro_rango_operativo,144144,91.19
1,faltante,13918,8.80
2,superior_a_120,14,0.01


Registros de edad que requieren revisión: 13932
Registros conservados: 158076


## 8. Control de coherencia de la limpieza
Confirmamos que no eliminamos filas ni alteramos columnas originales. Comprobamos que la edad derivada solo tiene enteros de 0 a 120 y que los casos no válidos quedan nulos en la nueva variable.

In [7]:
for nombre, original_df in TABLAS.items():
    nuevo = limpios[nombre]
    assert len(nuevo) == len(original_df), f'Cambió la cantidad de filas: {nombre}'
    assert nuevo[original_df.columns].equals(original_df), f'Se modificaron datos originales: {nombre}'
assert df_edad['edad_validada'].dropna().between(0, 120).all()
assert df_edad.loc[~valida, 'edad_validada'].isna().all()
assert int(resumen_edad['filas'].sum()) == len(df_edad)
print('VALIDACIONES SUPERADAS: filas y columnas originales intactas; edad derivada validada.')

VALIDACIONES SUPERADAS: filas y columnas originales intactas; edad derivada validada.


## 9. Exportación de tablas de trabajo y evidencias
Guardamos los cinco CSV de salida con los mismos registros. Solo `PersonasLyFporEdad_SSC` añade `edad_validada` y `estado_validacion_edad`. No denominamos estos archivos «sin duplicados», porque los casos no han sido eliminados.

In [8]:
impacto = []
for nombre, df in limpios.items():
    salida_csv = SALIDA / f'{nombre}_etapa3.csv'
    df.to_csv(salida_csv, index=False, encoding='utf-8-sig')
    comprobacion = pd.read_csv(salida_csv, dtype=str, encoding='utf-8-sig', low_memory=False)
    assert len(comprobacion) == len(TABLAS[nombre])
    impacto.append({
        'tabla': nombre,
        'filas_recibidas': len(TABLAS[nombre]),
        'filas_exportadas': len(df),
        'filas_eliminadas': len(TABLAS[nombre]) - len(df),
        'columnas_originales': len(TABLAS[nombre].columns),
        'columnas_exportadas': len(df.columns),
        'archivo': salida_csv.name
    })
impacto_df = pd.DataFrame(impacto)
display(impacto_df)
impacto_df.to_csv(SALIDA / 'impacto_etapa3_SSC.csv', index=False, encoding='utf-8-sig')
assert (impacto_df['filas_eliminadas'] == 0).all()

,tabla,filas_recibidas,filas_exportadas,filas_eliminadas,columnas_originales,columnas_exportadas,archivo
0,HechosTransito_SSC,134078,134078,0,27,27,HechosTransito_SSC_etapa3.csv
1,PersonasLyFporEdad_SSC,158076,158076,0,24,26,PersonasLyFporEdad_SSC_etapa3.csv
2,PersonasLyFporSexo_SSC,144043,144043,0,25,25,PersonasLyFporSexo_SSC_etapa3.csv
3,PersonasLyFporTipoPersona_SSC,107794,107794,0,25,25,PersonasLyFporTipoPersona_SSC_etapa3.csv
4,VehiculosInvolucrados_SSC,215078,215078,0,27,27,VehiculosInvolucrados_SSC_etapa3.csv


## 10. Bitácora de decisiones y pendientes
Generamos un registro legible de reglas aplicadas, evidencia y decisiones pendientes. Evitamos registrar como «duplicados eliminados» los 3,498 casos que solo han sido detectados.

In [9]:
bitacora = pd.DataFrame([
    {'regla':'Conservar archivos originales', 'accion':'No sobrescribir CSV de Etapa 1', 'estado':'aplicada',
     'justificacion':'Trazabilidad y reproducibilidad'},
    {'regla':'Folio repetido', 'accion':'Reportar y conservar', 'estado':'aplicada',
     'justificacion':'Un evento puede tener múltiples personas o vehículos'},
    {'regla':'Folio SD', 'accion':'No usar como clave única', 'estado':'aplicada',
     'justificacion':'Identificador no informativo'},
    {'regla':'Duplicados exactos en PersonasLyFporEdad', 'accion':'Conservar y revisar granularidad',
     'estado':'pendiente de decisión de eliminación',
     'justificacion':'3,498 repeticiones adicionales detectadas; podrían representar personas distintas'},
    {'regla':'Edad fuera de 0–120', 'accion':'Conservar edad original; edad_validada nula',
     'estado':'aplicada',
     'justificacion':'Valor no coherente para análisis por edades'},
    {'regla':'Edad faltante/no numérica/decimal', 'accion':'Conservar original; edad_validada nula',
     'estado':'aplicada',
     'justificacion':'No imputar información desconocida'},
    {'regla':'Exclusiones de Etapa 1', 'accion':'Mantener evidencias separadas',
     'estado':'aplicada',
     'justificacion':'AV INSURGENTES no identifica una alcaldía'},
])
display(bitacora)
bitacora.to_csv(SALIDA / 'bitacora_reglas_etapa3_SSC.csv', index=False, encoding='utf-8-sig')

,regla,accion,estado,justificacion
0,Conservar archivos originales,No sobrescribir CSV de Etapa 1,aplicada,Trazabilidad y reproducibilidad
1,Folio repetido,Reportar y conservar,aplicada,Un evento puede tener múltiples personas o veh...
2,Folio SD,No usar como clave única,aplicada,Identificador no informativo
3,Duplicados exactos en PersonasLyFporEdad,Conservar y revisar granularidad,pendiente de decisión de eliminación,"3,498 repeticiones adicionales detectadas; pod..."
4,Edad fuera de 0–120,Conservar edad original; edad_validada nula,aplicada,Valor no coherente para análisis por edades
5,Edad faltante/no numérica/decimal,Conservar original; edad_validada nula,aplicada,No imputar información desconocida
6,Exclusiones de Etapa 1,Mantener evidencias separadas,aplicada,AV INSURGENTES no identifica una alcaldía


## 11. Conclusión del equipo
En esta etapa **aplicamos una transformación controlada** sobre la variable edad y mantuvimos intactas todas las filas de las cinco tablas SSC. Generamos evidencia de los registros con edades faltantes o fuera del rango operativo y documentamos los folios y duplicados que requieren revisión.

Los **3,498 duplicados adicionales** de la tabla `PersonasLyFporEdad_SSC` permanecen en el conjunto de trabajo porque aún no se ha demostrado que correspondan a la misma persona u observación. Evitamos así una eliminación que podría alterar el análisis de severidad.

**Pendientes para cerrar la limpieza:** revisar la unidad de observación y, si existe, un identificador de persona; decidir y justificar cualquier deduplicación; revisar la lista de edades extremas y actualizar la bitácora con las decisiones finales. No sumamos filas de tablas diferentes como si fueran eventos únicos.